# Credit Risk PD/LGD/EAD  -  Exploratory Analysis

This notebook walks through the end-to-end credit risk modeling pipeline:

1. **Data Generation**  -  synthetic 500K+ loan portfolio
2. **Feature Engineering**  -  100+ credit features, WoE/IV, KS validation
3. **PD Modeling**  -  Logistic Regression & XGBoost
4. **LGD Modeling**  -  Beta Regression & Two-Stage GBM
5. **EAD Modeling**  -  Cox Regression & CCF
6. **Validation**  -  Hosmer-Lemeshow, PSI, Binomial Back-test
7. **Regulatory Capital**  -  Basel III IRB Advanced, Stress Testing

In [ ]:
import sys, yaml
sys.path.insert(0, '..')

from main import CreditRiskPipeline

with open('../configs/config.yaml') as f:
    config = yaml.safe_load(f)

# Use smaller sample for notebook exploration
config['data']['n_samples'] = 50_000

pipeline = CreditRiskPipeline(config)

## Step 1  -  Generate Data

In [ ]:
df = pipeline.step_1_generate_data()
print(f'Shape: {df.shape}')
print(f'Default rate: {df["default_flag"].mean():.4f}')
df.head()

## Step 2  -  Feature Engineering

In [ ]:
df = pipeline.step_2_engineer_features()
print(f'Features after engineering: {df.shape[1]}')
print(f'\nTop IV features:')
for feat in pipeline.results['features']['top_iv_features'][:5]:
    print(f"  {feat['feature']}: IV={feat['iv']:.4f} ({feat['predictive_power']})")

## Step 3  -  PD Model

In [ ]:
pd_results = pipeline.step_3_train_pd_model()

print(f"Selected model: {pd_results['selected']}")
print(f"AUC:  {pd_results['metrics']['auc']}")
print(f"Gini: {pd_results['metrics']['gini']}")
print(f"KS:   {pd_results['metrics']['ks_statistic']}")

## Step 4-5  -  LGD & EAD Models

In [ ]:
lgd_results = pipeline.step_4_train_lgd_model()
print(f"LGD R2:   {lgd_results['r2']}")
print(f"LGD RMSE: {lgd_results['rmse']}")

ead_results = pipeline.step_5_train_ead_model()
print(f"EAD C-index: {ead_results['c_index']}")
print(f"EAD MAPE:    {ead_results['mape']}%")

## Step 6  -  Validation

In [ ]:
val = pipeline.step_6_validate_models()

print(f"Hosmer-Lemeshow: chi2={val['hosmer_lemeshow']['chi_squared']}, p={val['hosmer_lemeshow']['p_value']}")
print(f"PSI: {val['psi']['psi']} ({val['psi']['stability']})")
print(f"Backtest zone: {val['binomial_backtest']['traffic_light_zone']}")

## Step 7  -  Regulatory Capital & Stress Testing

In [ ]:
reg = pipeline.step_7_regulatory_capital()

print(f"Total EL:       ${reg['expected_loss']['total']:,.0f}")
print(f"Total RWA:      ${reg['rwa']['total']:,.0f}")
print(f"Capital Req:    ${reg['rwa']['capital_required']:,.0f}")
print(f"Economic Cap:   ${reg['economic_capital']['ec']:,.0f}")
print(f"\nStress scenarios:")
for name, data in reg['stress_test'].items():
    print(f"  {name}: PD={data['mean_pd']:.4f}, EL=${data['total_el']:,.0f}, Migration={data['pd_migration_pct']:+.1f}%")